In [1]:
# Data handling
import pandas as pd
import numpy as np

# Text preprocessing
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\girid_z5cdims\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\girid_z5cdims\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# Load datasets
train_df = pd.read_csv(r"D:\INTERNSHIPS\SENTIMENT-ANALYSIS-WITH-NLP\Train.csv")
valid_df = pd.read_csv(r"D:\INTERNSHIPS\SENTIMENT-ANALYSIS-WITH-NLP\Valid.csv")
test_df  = pd.read_csv(r"D:\INTERNSHIPS\SENTIMENT-ANALYSIS-WITH-NLP\Test.csv")

# Remove unwanted spaces from column names
train_df.columns = train_df.columns.str.strip()
valid_df.columns = valid_df.columns.str.strip()
test_df.columns  = test_df.columns.str.strip()

# Display dataset info
print("Training Data Shape:", train_df.shape)
print("Validation Data Shape:", valid_df.shape)
print("Test Data Shape:", test_df.shape)

train_df.head()


Training Data Shape: (40000, 2)
Validation Data Shape: (5000, 2)
Test Data Shape: (5000, 2)


,Review,Sentiment
0,I grew up (b. 1965) watching and loving the Th...,Negative
1,"When I put this movie in my DVD player, and sa...",Negative
2,Why do people who do not know what a particula...,Negative
3,Even though I have great interest in Biblical ...,Negative
4,Im a die hard Dads Army fan and nothing will e...,Positive


In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove numbers and special characters
    text = re.sub(r"[^a-z\s]", "", text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords and apply lemmatization
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)


In [5]:
# Clean review text
train_df["clean_review"] = train_df["Review"].apply(clean_text)
valid_df["clean_review"] = valid_df["Review"].apply(clean_text)
test_df["clean_review"]  = test_df["Review"].apply(clean_text)


In [6]:
# Convert sentiment labels to numeric values
# Positive → 1, Negative → 0
train_df["Sentiment"] = train_df["Sentiment"].map({"Positive": 1, "Negative": 0})
valid_df["Sentiment"] = valid_df["Sentiment"].map({"Positive": 1, "Negative": 0})
test_df["Sentiment"]  = test_df["Sentiment"].map({"Positive": 1, "Negative": 0})


In [7]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Fit on training data
X_train = tfidf.fit_transform(train_df["clean_review"])
X_valid = tfidf.transform(valid_df["clean_review"])
X_test  = tfidf.transform(test_df["clean_review"])

# Target labels
y_train = train_df["Sentiment"]
y_valid = valid_df["Sentiment"]
y_test  = test_df["Sentiment"]


In [8]:
# Initialize Logistic Regression
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train, y_train)

print("Model training completed successfully!")


Model training completed successfully!


In [9]:
# Predict on validation data
valid_pred = model.predict(X_valid)

# Validation accuracy
val_accuracy = accuracy_score(y_valid, valid_pred)
print("Validation Accuracy:", val_accuracy)

print("\nClassification Report (Validation):")
print(classification_report(y_valid, valid_pred))


Validation Accuracy: 0.8798

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.89      0.86      0.88      2486
           1       0.87      0.90      0.88      2514

    accuracy                           0.88      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.88      0.88      0.88      5000



In [10]:
# Predict on test data
test_pred = model.predict(X_test)

# Test accuracy
test_accuracy = accuracy_score(y_test, test_pred)
print("Test Accuracy:", test_accuracy)

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred))


Test Accuracy: 0.8854

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.89      0.88      0.88      2495
           1       0.88      0.89      0.89      2505

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000



In [11]:
print("Confusion Matrix (Test Data):")
print(confusion_matrix(y_test, test_pred))


Confusion Matrix (Test Data):
[[2191  304]
 [ 269 2236]]


In [12]:
def predict_sentiment(review):
    cleaned_review = clean_text(review)
    review_vector = tfidf.transform([cleaned_review])
    
    prediction = model.predict(review_vector)[0]
    confidence = max(model.predict_proba(review_vector)[0]) * 100
    
    sentiment = "Positive 😊" if prediction == 1 else "Negative 😠"
    
    return sentiment, confidence



In [14]:
# 20 sample reviews for testing the model
sample_reviews = [
    "The movie was absolutely fantastic and I loved every scene",
    "Worst movie ever, completely wasted my time",
    "Amazing performance by the lead actor",
    "The storyline was boring and predictable",
    "I enjoyed the film, it was entertaining",
    "Terrible acting and poor direction",
    "One of the best movies I have seen this year",
    "The movie was too long and very dull",
    "Excellent visuals and great background music",
    "I did not like the movie at all",
    "The plot was interesting and engaging",
    "Bad screenplay and weak characters",
    "The film exceeded my expectations",
    "Not worth watching, very disappointing",
    "Outstanding direction and brilliant acting",
    "The movie failed to impress me",
    "A wonderful cinematic experience",
    "The story lacked depth and emotion",
    "Highly recommended movie",
    "The movie was horrible and annoying"
]

# Predict sentiment for each review
for i, review in enumerate(sample_reviews, start=1):
    sentiment, confidence = predict_sentiment(review)
    print(f"Review {i}: {review}")
    print(f"Prediction: {sentiment} | Confidence: {confidence:.2f}%")
    print("-" * 70)


Review 1: The movie was absolutely fantastic and I loved every scene
Prediction: Positive 😊 | Confidence: 97.83%
----------------------------------------------------------------------
Review 2: Worst movie ever, completely wasted my time
Prediction: Negative 😠 | Confidence: 99.93%
----------------------------------------------------------------------
Review 3: Amazing performance by the lead actor
Prediction: Positive 😊 | Confidence: 96.99%
----------------------------------------------------------------------
Review 4: The storyline was boring and predictable
Prediction: Negative 😠 | Confidence: 99.63%
----------------------------------------------------------------------
Review 5: I enjoyed the film, it was entertaining
Prediction: Positive 😊 | Confidence: 99.44%
----------------------------------------------------------------------
Review 6: Terrible acting and poor direction
Prediction: Negative 😠 | Confidence: 99.93%
----------------------------------------------------------------